In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import qiskit_metal as metal
from qiskit_metal import Dict, designs, MetalGUI
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
import numpy as np
from collections import OrderedDict
from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors
from qiskit_metal.qlibrary.tlines.framed_path import RouteFramed
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.lumped.cap_n_interdigital import CapNInterdigital

In [3]:
design = designs.DesignPlanar(overwrite_enabled=True)

In [4]:
gui = MetalGUI(design)
design.variables['cpw_width']='10um'
design.variables['cpw_gap']='6um'

# Calculos das capacitancias

In [20]:
l = 1000
d = np.linspace(0, 1000, 100)

a0 = 1.62129e-7
a1 = 0.100845
b0 = 0.0592123
b1 = -2.29999e-6
m0 = 0.000600864
m1 = -2.1275e-6
c0 = -0.201206
c1 = 0.00156082
A = a0 + a1*l
B = b0 + b1*l
M = m0 + m1*l
C = c0 + c1*l

C = A*np.exp(-B*d) + M*d + C

plt.plot(d, C)

In [37]:
def Cap(d, l):
    """d (um) - distancia do gap entre os dois metais
        l (um) - comprimento dos metais que ficam paralelos"""
    a0 = 1.62129e-7
    a1 = 0.100845
    b0 = 0.0592123
    b1 = -2.29999e-6
    m0 = 0.000600864
    m1 = -2.1275e-6
    c0 = -0.201206
    c1 = 0.00156082
    A = a0 + a1*l
    B = b0 + b1*l
    M = m0 + m1*l
    C = c0 + c1*l
    
    C = A*np.exp(-B*d) + M*d + C
    return C

Cap(500, 2000)

1.0933660002791064

# Definindo funções

In [21]:
def ressonador_cima(resonator_number, connection1, connection2):
    """
    resonator_number - numero do ressonador, para nomerar o ressonador
    connection = [pos_x, pos_y] em milimetros"""

    OpenToGround(design, f'OTG{resonator_number}_1', options=Dict(
        pos_x=f'{connection1[0]}mm',
        pos_y=f'{connection1[1]}mm',
        orientation='90',
        #width='10um',
        #gap='6um',
        #termination_gap='12um'
    ))

    OpenToGround(design, f'OTG{resonator_number}_2', options=Dict(
        pos_x=f'{connection2[0]}mm',
        pos_y=f'{connection2[1]}mm',
        orientation='90',
        #width='10um',
        #gap='6um',
        #termination_gap='12um'
    ))

    return RouteFramed(design, f'RES{resonator_number}', options=Dict(
        pin_inputs = Dict(
            start_pin = Dict(
                component = f'OTG{resonator_number}_1',
                pin = 'open'
            ),
            end_pin = Dict(
                component = f'OTG{resonator_number}_2',
                pin = 'open'
            )
        ),
        #total_length='10mm',
        lead=Dict(start_straight='4.15mm', end_straight='4.15mm'),
        fillet='300um',
        hfss_wire_bonds = True
    ))

In [25]:
def ressonador_baixo(resonator_number, connection1, connection2):
    """
    resonator_number - numero do ressonador, para nomerar o ressonador
    connection = [pos_x, pos_y] em milimetros"""

    OpenToGround(design, f'OTG{resonator_number}_1', options=Dict(
        pos_x=f'{connection1[0]}mm',
        pos_y=f'{connection1[1]}mm',
        orientation='-90',
        #width='10um',
        #gap='6um',
        #termination_gap='12um'
    ))

    OpenToGround(design, f'OTG{resonator_number}_2', options=Dict(
        pos_x=f'{connection2[0]}mm',
        pos_y=f'{connection2[1]}mm',
        orientation='-90',
        #width='10um',
        #gap='6um',
        #termination_gap='12um'
    ))

    return RouteFramed(design, f'RES{resonator_number}', options=Dict(
        pin_inputs = Dict(
            start_pin = Dict(
                component = f'OTG{resonator_number}_1',
                pin = 'open'
            ),
            end_pin = Dict(
                component = f'OTG{resonator_number}_2',
                pin = 'open'
            )
        ),
        total_length='10mm',
        lead=Dict(start_straight='4.15mm', end_straight='4.15mm'),
        fillet='300um',
        hfss_wire_bonds = True
    ))

In [45]:
x = 2.8 # distancia dos ressonadores na horizontal
y = 5.8 # distancia dos ressonadores na vertical
res1 = ressonador_cima(1, [0, 0], [2, 0])
res2 = ressonador_cima(2, [1*x, 0], [1*x+2, 0])
res3 = ressonador_cima(3, [2*x, 0], [2*x+2, 0])
res4 = ressonador_cima(4, [3*x, 0], [3*x+2, 0])
res5 = ressonador_cima(5, [4*x, 0], [4*x+2, 0])
res6 = ressonador_cima(6, [5*x, 0], [5*x+2, 0])
res7 = ressonador_cima(7, [6*x, 0], [6*x+2, 0])
res8 = ressonador_cima(8, [7*x, 0], [7*x+2, 0])

res9 = ressonador_baixo(9, [1.4, -3], [1.4+2, -3])
res10 = ressonador_baixo(10, [1.4+1*x, -3], [1.4+1*x+2, -3])
res11 = ressonador_baixo(11, [1.4+2*x, -3], [1.4+2*x+2, -3])
res12 = ressonador_baixo(12, [1.4+3*x, -3], [1.4+3*x+2, -3])
res13 = ressonador_baixo(13, [1.4+4*x, -3], [1.4+4*x+2, -3])
res14 = ressonador_baixo(14, [1.4+5*x, -3], [1.4+5*x+2, -3])
res15 = ressonador_baixo(15, [1.4+6*x, -3], [1.4+6*x+2, -3])

res16 = ressonador_cima(16, [1.4+0*x, y], [1.4+0*x+2, y])
res17 = ressonador_cima(17, [1.4+1*x, y], [1.4+1*x+2, y])
res18 = ressonador_cima(18, [1.4+2*x, y], [1.4+2*x+2, y])
res19 = ressonador_cima(19, [1.4+3*x, y], [1.4+3*x+2, y])
res20 = ressonador_cima(20, [1.4+4*x, y], [1.4+4*x+2, y])
res21 = ressonador_cima(21, [1.4+5*x, y], [1.4+5*x+2, y])
res22 = ressonador_cima(22, [1.4+6*x, y], [1.4+6*x+2, y])

res23 = ressonador_baixo(23, [0, -3+y], [2, -3+y])
res24 = ressonador_baixo(24, [1*x, -3+y], [1*x+2, -3+y])
res25 = ressonador_baixo(25, [2*x, -3+y], [2*x+2, -3+y])
res26 = ressonador_baixo(26, [3*x, -3+y], [3*x+2, -3+y])
res27 = ressonador_baixo(27, [4*x, -3+y], [4*x+2, -3+y])
res28 = ressonador_baixo(28, [5*x, -3+y], [5*x+2, -3+y])
res29 = ressonador_baixo(29, [6*x, -3+y], [6*x+2, -3+y])
res30 = ressonador_baixo(30, [7*x, -3+y], [7*x+2, -3+y])

res31 = ressonador_cima(31, [0*x, 2*y], [0*x+2, 2*y])
res32 = ressonador_cima(32, [1*x, 2*y], [1*x+2, 2*y])
res33 = ressonador_cima(33, [2*x, 2*y], [2*x+2, 2*y])
res34 = ressonador_cima(34, [3*x, 2*y], [3*x+2, 2*y])
res35 = ressonador_cima(35, [4*x, 2*y], [4*x+2, 2*y])
res36 = ressonador_cima(36, [5*x, 2*y], [5*x+2, 2*y])
res37 = ressonador_cima(37, [6*x, 2*y], [6*x+2, 2*y])
res38 = ressonador_cima(38, [7*x, 2*y], [7*x+2, 2*y])


gui.rebuild()
gui.autoscale()

In [20]:
design.delete_all_components()